In [1]:
import subprocess
subprocess.run(["pip", "install", "google-generativeai", "beautifulsoup4", "requests"])


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', 'google-generativeai', 'beautifulsoup4', 'requests'], returncode=0)

In [5]:
from dotenv import load_dotenv, find_dotenv
import os
import csv
import time
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai

# Loading both keys
load_dotenv(find_dotenv(), override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# ensure it worked
print("Setup complete")

Setup complete


In [7]:
def scrape_website(url):
    """
    Visits a website and extracts the text content.
    Returns the text so Gemini can read it.
    """
    if not url or url == "Not listed":
        return None
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        
        # Parse the HTML and extract just the text
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Remove navigation, scripts, and styling
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        
        text = soup.get_text(separator=" ", strip=True)
        
        return text[:25000]
    
    except Exception as e:
        print(f" Could not scrape {url}: {e}")
        return None

In [8]:
urban_farm_def = """
An urban farm qualifies for include if it meets all of these:
- Physically located in Harris County, Texas 
- Sells agricultural products directly

Include these types if they sell products:
- Urban farms, market gardens, microfarms
- Hydroponic, aquaponic, greenhouse, indoor, vertical farms
- Microgreens operations
- CSA farms
- Flower farms
- Small livestock, egg, honey, mushroom, or nursery operations

Exclude these:
- Community gardens with no sales
- Educational gardens with no sales  
- Donation-only farms
- Farmers markets, food hubs, distributors, or resellers 
  (unless they are also producers)
- Farms outside Harris County
- Entities with no usable location

Mark as review if:
- Unclear whether they sell products
- Unclear if they are in Harris County
- Website doesn't have enough information
- Only has a Facebook/Instagram page (no real website)
"""

def filter_with_gemini(name, address, website_text, url):
    """
    Sends farm info + website content to Gemini.
    Gemini decides Include/Exclude/Review based on your definition.
    """
    
    if not website_text:
        return {
            "status": "Review",
            "reason": "No website available for verification"
        }
    
    prompt = f"""
    You are helping build a database of urban farms in Harris County, Texas.
    
    Here is the definition of what qualifies:
    {urban_farm_def}
    
    Here is the farm to evaluate:
    Name: {name}
    Address: {address}
    Website: {url}
    
    Website content:
    {website_text}
    
    Based on the website content and definition above, evaluate this farm.
    
    Respond ONLY in this exact format with no extra text:
    STATUS: [Include/Exclude/Review]
    REASON: [One sentence explanation]
    FARM_TYPE: [e.g. Urban Farm, Flower Farm, Honey Operation, Farmers Market, etc.]
    SELLS_PRODUCTS: [Yes/No/Unclear]
    """
    
    try:
        response = model.generate_content(prompt)
        text = response.text.strip()
        
        # Parse the response
        lines = text.split("\n")
        result = {}
        for line in lines:
            if line.startswith("STATUS:"):
                result["status"] = line.replace("STATUS:", "").strip()
            elif line.startswith("REASON:"):
                result["reason"] = line.replace("REASON:", "").strip()
            elif line.startswith("FARM_TYPE:"):
                result["farm_type"] = line.replace("FARM_TYPE:", "").strip()
            elif line.startswith("SELLS_PRODUCTS:"):
                result["sells_products"] = line.replace("SELLS_PRODUCTS:", "").strip()
        
        return result
    
    except Exception as e:
        print(f"  Gemini error for {name}: {e}")
        return {
            "status": "Review",
            "reason": f"Error during analysis: {e}",
        }

In [9]:
def process_farms(input_file="harris_county_farms.csv", 
                  output_file="filtered_harris_county_farms.csv"):
    
    results = []
    
    # read CSV
    with open(input_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        farms = list(reader)
    
    print(f"Processing {len(farms)} farms...\n")
    
    for i, farm in enumerate(farms):
        name = farm.get("Name", "")
        address = farm.get("Address", "")
        url = farm.get("Website", "Not listed")
        lat = farm.get("Latitude", "")
        lng = farm.get("Longitude", "")
        place_id = farm.get("Place ID", "")
        maps_link = f"https://www.google.com/maps/place/?q=place_id:{place_id}"
        
        print(f"[{i+1}/{len(farms)}] Processing: {name}")
        
        # scrape website
        if url and url != "Not listed":
            print(f"  Scraping: {url}")
            website_text = scrape_website(url)
        else:
            website_text = None
            print(f"  No website listed")
        
        # filter 
        print(f"  Analyzing with Gemini...")
        gemini_result = filter_with_gemini(name, address, website_text, url)
        
        # store data in dic
        results.append({
            "Name": name,
            "Address": address,
            "Latitude": lat,
            "Longitude": lng,
            "Website": url,
            "Google Maps Link": maps_link,
            "Status": gemini_result.get("status", "Review"),
            "Reason": gemini_result.get("reason", ""),
            "Farm Type": gemini_result.get("farm_type", ""),
            "Sells Products": gemini_result.get("sells_products", "")
        })
        
        print(f"  Status: {gemini_result.get('status')} — {gemini_result.get('reason')}\n")
        
        time.sleep(4)
    
    # put in new CSV
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "Name", "Address", "Latitude", "Longitude",
            "Website", "Google Maps Link", "Status",
            "Reason", "Farm Type", "Sells Products"
        ])
        writer.writeheader()
        writer.writerows(results)
    
    # Print summary
    statuses = [r["Status"] for r in results]
    print(f"\n{'='*50}")
    print(f" Results saved to {output_file}")
    print(f"{'='*50}")
    print(f"Include:  {statuses.count('Include')}")
    print(f"Exclude:  {statuses.count('Exclude')}")
    print(f"Review:   {statuses.count('Review')}")
    print(f"Total:    {len(results)}")

process_farms()

Processing 171 farms...

[1/171] Processing: Bhakti Urban Farm
  Scraping: https://bhaktiurbanfarm.org/
  Analyzing with Gemini...
  Status: Review — The website content is inaccessible, making it unclear if they sell products or provide enough information for a full evaluation.

[2/171] Processing: Hope Farms Urban Agricultural Showcase and Training Center - Recipe for Success
  Scraping: https://hopefarmshtx.org/
  Analyzing with Gemini...
  Status: Review — The website focuses on education, training, and community engagement, but it is unclear if agricultural products are sold directly to the public.

[3/171] Processing: Finca Tres Robles
  Scraping: http://www.smallplaces.org/
  Analyzing with Gemini...
  Status: Review — The website content describes distributing produce boxes and nourishing families through gifts and programs, but it does not clearly state or imply that the farm sells agricultural products directly.

[4/171] Processing: Montrose Urban Food Farm
  No website liste

KeyboardInterrupt: 